In [2]:
# sqlite3 is built into Python — no installation needed
import sqlite3
import pandas as pd

In [13]:
# connect() creates the database file if it doesn't exist
# use ':memory:' instead of a filename to create a temporary in-memory database
conn = sqlite3.connect('students.db')

# cursor is used to execute SQL commands
cursor = conn.cursor()

In [4]:
# CREATE TABLE — define the structure of the table
# IF NOT EXISTS prevents an error if the table already exists
cursor.execute('''
    CREATE TABLE IF NOT EXISTS students (
        id      INTEGER PRIMARY KEY AUTOINCREMENT,
        name    TEXT    NOT NULL,
        grade   INTEGER,
        score   REAL
    )
''')

# commit() saves the changes to the file
conn.commit()
print('Table created')

Table created


In [5]:
# INSERT rows using parameterized queries
# use ? placeholders instead of string formatting to avoid SQL injection
students = [
    ('Ahmed',  3, 92.5),
    ('Sara',   2, 88.0),
    ('Omar',   3, 75.3),
    ('Nadia',  1, 95.1),
]

# executemany() inserts multiple rows at once
cursor.executemany('INSERT INTO students (name, grade, score) VALUES (?, ?, ?)', students)
conn.commit()
print(f'{cursor.rowcount} rows inserted')

4 rows inserted


In [6]:
# SELECT all rows from the table
cursor.execute('SELECT * FROM students')

# fetchall() returns all rows as a list of tuples
rows = cursor.fetchall()

for row in rows:
    print(row)

(1, 'Ahmed', 3, 92.5)
(2, 'Sara', 2, 88.0)
(3, 'Omar', 3, 75.3)
(4, 'Nadia', 1, 95.1)


In [7]:
# read directly into a pandas DataFrame using pd.read_sql()
# this is the most convenient way to work with SQL results in pandas
df = pd.read_sql('SELECT * FROM students', conn)
print(df)

   id   name  grade  score
0   1  Ahmed      3   92.5
1   2   Sara      2   88.0
2   3   Omar      3   75.3
3   4  Nadia      1   95.1


In [8]:
# filter rows with WHERE clause
df_grade3 = pd.read_sql('SELECT * FROM students WHERE grade = 3', conn)
print(df_grade3)

   id   name  grade  score
0   1  Ahmed      3   92.5
1   3   Omar      3   75.3


In [9]:
# UPDATE — modify existing rows
cursor.execute('UPDATE students SET score = 99.0 WHERE name = ?', ('Ahmed',))
conn.commit()

df = pd.read_sql('SELECT * FROM students', conn)
print(df)

   id   name  grade  score
0   1  Ahmed      3   99.0
1   2   Sara      2   88.0
2   3   Omar      3   75.3
3   4  Nadia      1   95.1


In [14]:
# DELETE — remove rows
cursor.execute('DELETE FROM students WHERE score < 80')
conn.commit()

df = pd.read_sql('SELECT * FROM students', conn)
print(df)

   id   name  grade  score
0   1  Ahmed      3   99.0
1   2   Sara      2   88.0
2   4  Nadia      1   95.1


In [15]:
# always close the connection when done to release the file lock
conn.close()
print('Connection closed')

Connection closed
